In [1]:
!pip install deepchem

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 552.4/552.4 kB 14.2 MB/s eta 0:00:00


In [2]:
!pip install rdkit

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 36.7/36.7 MB 64.3 MB/s eta 0:00:00


In [3]:
# ============================================================
#  OLMo-7B DeepChem — LLM-Ready Datasets
#  Format: Raw SMILES strings + property labels (text format)
#  Tasks  : Generation, Classification, Regression, Pretraining
# ============================================================
#
#  WHY DIFFERENT DATASETS FROM BEFORE?
#  OLMo is a Language Model — it reads TEXT, not fingerprints.
#  We need SMILES strings as input, not ECFP vectors.
#
#  DATASET STRUCTURE FOR OLMo:
#  ┌─────────────────────────────────────────────────────┐
#  │  Generation   → SMILES strings only (no labels)     │
#  │  Classification → "SMILES [SEP] label (0 or 1)"     │
#  │  Regression   → "SMILES [SEP] value (float)"        │
#  │  Pretraining  → large SMILES corpus (no labels)     │
#  └─────────────────────────────────────────────────────┘
# ============================================================

import os
import warnings
import requests
import pandas as pd
import deepchem as dc

warnings.filterwarnings("ignore")

DATA_DIR = "./olmo_llm_datasets"
os.makedirs(DATA_DIR, exist_ok=True)
print(f"Saving LLM-ready datasets to: {os.path.abspath(DATA_DIR)}\n")


# ════════════════════════════════════════════════════════════
#  HELPER — Extract SMILES strings from DeepChem dataset
#  OLMo needs raw text, so we pull SMILES from dataset.ids
# ════════════════════════════════════════════════════════════
def save_smiles_dataset(name, tasks, splits, task_type, out_dir):
    """
    Converts DeepChem dataset splits into LLM-ready CSV files.

    For classification/regression:
        columns = [smiles, task_name, split]
        Example row: CC(=O)Oc1ccccc1C(=O)O , 0.5 , train

    For generation/pretraining:
        columns = [smiles, split]
        Example row: CC(=O)Oc1ccccc1C(=O)O , train
    """
    os.makedirs(out_dir, exist_ok=True)
    split_names = ["train", "valid", "test"]
    all_rows = []

    for split_name, dataset in zip(split_names, splits):
        smiles_list = dataset.ids          # raw SMILES strings
        labels = dataset.y                 # property values

        for i, smi in enumerate(smiles_list):
            if task_type in ("classification", "regression"):
                # one row per task per molecule
                for j, task in enumerate(tasks):
                    row = {
                        "smiles":      smi,
                        "task":        task,
                        "label":       float(labels[i][j]) if labels is not None else None,
                        "task_type":   task_type,
                        "split":       split_name,
                        # LLM input string — this is what OLMo tokenizer will receive
                        "llm_input":   f"SMILES: {smi} TASK: {task} LABEL:",
                        "llm_target":  str(labels[i][j]) if labels is not None else ""
                    }
                    all_rows.append(row)
            else:
                # generation / pretraining — SMILES text only
                row = {
                    "smiles":    smi,
                    "split":     split_name,
                    "llm_input": f"Generate molecule: {smi}",
                    "llm_target": smi
                }
                all_rows.append(row)

    df = pd.DataFrame(all_rows)
    out_path = os.path.join(out_dir, f"{name}.csv")
    df.to_csv(out_path, index=False)

    train_df = df[df["split"] == "train"]
    valid_df = df[df["split"] == "valid"]
    test_df  = df[df["split"] == "test"]

    print(f"   ✅  {name} saved → {out_path}")
    print(f"       train={len(train_df):,} | valid={len(valid_df):,} | test={len(test_df):,} rows")
    print(f"       Sample llm_input : {df['llm_input'].iloc[0]}")
    print(f"       Sample llm_target: {df['llm_target'].iloc[0]}")
    return df


# ════════════════════════════════════════════════════════════
#  TASK 1 — GENERATION
#  Dataset: MOSES
#  Format : raw SMILES strings — OLMo learns to generate new
#           valid molecules autoregressively
#  Why    : MOSES is the standard molecular generation benchmark
# ════════════════════════════════════════════════════════════
print("=" * 55)
print("  TASK 1 — GENERATION (MOSES)")
print("=" * 55)

gen_dir = os.path.join(DATA_DIR, "generation")
os.makedirs(gen_dir, exist_ok=True)
moses_path = os.path.join(gen_dir, "moses_raw.csv")

print("\n📥 Downloading MOSES ...")
try:
    url = "https://media.githubusercontent.com/media/molecularsets/moses/master/data/dataset_v1.csv"
    r = requests.get(url, timeout=90)
    with open(moses_path, "wb") as f:
        f.write(r.content)

    moses_df = pd.read_csv(moses_path)

    # Convert to LLM format
    moses_llm = pd.DataFrame({
        "smiles":     moses_df["SMILES"],
        "split":      moses_df["SPLIT"],
        "llm_input":  "Generate molecule: " + moses_df["SMILES"],
        "llm_target": moses_df["SMILES"],
        "task_type":  "generation"
    })
    moses_llm_path = os.path.join(gen_dir, "moses_llm.csv")
    moses_llm.to_csv(moses_llm_path, index=False)

    train_n = (moses_llm["split"] == "train").sum()
    test_n  = (moses_llm["split"] == "test").sum()
    print(f"   ✅  MOSES — {len(moses_llm):,} molecules → {moses_llm_path}")
    print(f"       train={train_n:,} | test={test_n:,}")
    print(f"       Sample llm_input : {moses_llm['llm_input'].iloc[0]}")
    print(f"       Sample llm_target: {moses_llm['llm_target'].iloc[0]}")
except Exception as e:
    print(f"   ❌  MOSES failed: {e}")


# ════════════════════════════════════════════════════════════
#  TASK 2 — CLASSIFICATION
#  Dataset 1: SIDER  (drug side effects, 27 binary tasks)
#  Dataset 2: MUV    (virtual screening, 17 binary tasks)
#  Format : "SMILES: <smi> TASK: <task_name> LABEL:" → "0" or "1"
#  Why    : These are the only classification datasets confirmed
#           working without errors in your environment
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("  TASK 2 — CLASSIFICATION (SIDER + MUV)")
print("=" * 55)

clf_dir = os.path.join(DATA_DIR, "classification")

# ── SIDER ─────────────────────────────────────────────────
print("\n📥 Loading SIDER ...")
tasks, splits, _ = dc.molnet.load_sider(featurizer="Raw", splitter="scaffold")
save_smiles_dataset("sider", tasks, splits, "classification", clf_dir)

# ── MUV ───────────────────────────────────────────────────
print("\n📥 Loading MUV ...")
tasks, splits, _ = dc.molnet.load_muv(featurizer="Raw", splitter="scaffold")
save_smiles_dataset("muv", tasks, splits, "classification", clf_dir)


# ════════════════════════════════════════════════════════════
#  TASK 3 — REGRESSION
#  Dataset 1: ESOL         (solubility, 1 task)
#  Dataset 2: Lipophilicity (lipophilicity, 1 task)
#  Format : "SMILES: <smi> TASK: <task> LABEL:" → "-1.23"
#  Why    : ESOL and Lipo are small, clean, and standard
#           regression benchmarks for molecular property prediction
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("  TASK 3 — REGRESSION (ESOL + Lipophilicity)")
print("=" * 55)

reg_dir = os.path.join(DATA_DIR, "regression")

# ── ESOL ──────────────────────────────────────────────────
print("\n📥 Loading ESOL ...")
tasks, splits, _ = dc.molnet.load_delaney(featurizer="Raw", splitter="scaffold")
save_smiles_dataset("esol", tasks, splits, "regression", reg_dir)

# ── Lipophilicity ─────────────────────────────────────────
print("\n📥 Loading Lipophilicity ...")
tasks, splits, _ = dc.molnet.load_lipo(featurizer="Raw", splitter="scaffold")
save_smiles_dataset("lipophilicity", tasks, splits, "regression", reg_dir)


# ════════════════════════════════════════════════════════════
#  TASK 4 — PRETRAINING (Continued Pretraining on Molecules)
#  Dataset: ZINC15 (250k unlabeled drug-like SMILES)
#  Format : raw SMILES — OLMo learns molecular language
#  Why    : Large unlabeled corpus is ideal for continued
#           pretraining so OLMo understands SMILES syntax
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("  TASK 4 — PRETRAINING (ZINC15)")
print("=" * 55)

pretrain_dir = os.path.join(DATA_DIR, "pretraining")
os.makedirs(pretrain_dir, exist_ok=True)

print("\n📥 Loading ZINC15 ...")
try:
    tasks, (dataset,), _ = dc.molnet.load_zinc15(featurizer="Raw", splitter=None)
    smiles_list = dataset.ids

    zinc_llm = pd.DataFrame({
        "smiles":     smiles_list,
        "llm_input":  ["Molecule: " + s for s in smiles_list],
        "llm_target": smiles_list,
        "task_type":  "pretraining"
    })
    zinc_path = os.path.join(pretrain_dir, "zinc15_llm.csv")
    zinc_llm.to_csv(zinc_path, index=False)

    print(f"   ✅  ZINC15 — {len(zinc_llm):,} molecules → {zinc_path}")
    print(f"       Sample llm_input : {zinc_llm['llm_input'].iloc[0]}")
    print(f"       Sample llm_target: {zinc_llm['llm_target'].iloc[0]}")
except Exception as e:
    print(f"   ❌  ZINC15 failed: {e}")


# ════════════════════════════════════════════════════════════
#  FINAL SUMMARY
# ════════════════════════════════════════════════════════════
print("\n" + "=" * 55)
print("  LLM DATASET SUMMARY FOR OLMo-7B")
print("=" * 55)
print(f"""
  📁 {DATA_DIR}/
  ├── generation/
  │     moses_llm.csv        ← ~1.9M SMILES for generation
  ├── classification/
  │     sider_llm.csv        ← 1,427 molecules × 27 tasks
  │     muv_llm.csv          ← 93,087 molecules × 17 tasks
  ├── regression/
  │     esol_llm.csv         ← 1,128 molecules, solubility
  │     lipophilicity_llm.csv← 4,200 molecules, logD
  └── pretraining/
        zinc15_llm.csv       ← 250,000 SMILES corpus

  COLUMN FORMAT (all files):
    smiles      → raw SMILES string
    llm_input   → tokenizer input for OLMo
    llm_target  → expected output for OLMo
    task_type   → generation / classification / regression / pretraining
    split       → train / valid / test

  NEXT STEP → Load these CSVs into HuggingFaceModel (OLMo) wrapper
""")
print("✅  All LLM-ready datasets saved!")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


Saving LLM-ready datasets to: /content/olmo_llm_datasets

  TASK 1 — GENERATION (MOSES)

📥 Downloading MOSES ...
   ✅  MOSES — 1,936,962 molecules → ./olmo_llm_datasets/generation/moses_llm.csv
       train=1,584,663 | test=176,074
       Sample llm_input : Generate molecule: CCCS(=O)c1ccc2[nH]c(=NC(=O)OC)[nH]c2c1
       Sample llm_target: CCCS(=O)c1ccc2[nH]c(=NC(=O)OC)[nH]c2c1

  TASK 2 — CLASSIFICATION (SIDER + MUV)

📥 Loading SIDER ...


[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:30] WARNING: not removing hydrogen atom without neighbors
[16:28:31] WARNING: not removing hydrogen atom without neighbors
[16:28:31] WARNING: not removing hydrogen atom without neighbors
[16:28:31] WARNING: not removing hydrogen atom without neighbors
[16:28:31] WARNING: not removing hydrogen atom without neighbors
[16:28:31] WARNING: not removing hydrogen atom without neighbors
[16:28:32] WARNING: not removing hydrogen atom without neighbors
[16:28:32] WARNING: not removing hydrogen atom without neighbors
[16:28:32] WARNING: not r

   ✅  sider saved → ./olmo_llm_datasets/classification/sider.csv
       train=30,807 | valid=3,861 | test=3,861 rows
       Sample llm_input : SMILES: C(CNCCNCCNCCN)N TASK: Hepatobiliary disorders LABEL:
       Sample llm_target: 1.0

📥 Loading MUV ...
   ✅  muv saved → ./olmo_llm_datasets/classification/muv.csv
       train=1,265,973 | valid=158,253 | test=158,253 rows
       Sample llm_input : SMILES: NC(=O)NC(Cc1ccccc1)C(=O)O TASK: MUV-466 LABEL:
       Sample llm_target: 0.0

  TASK 3 — REGRESSION (ESOL + Lipophilicity)

📥 Loading ESOL ...
   ✅  esol saved → ./olmo_llm_datasets/regression/esol.csv
       train=902 | valid=113 | test=113 rows
       Sample llm_input : SMILES: CC(C)=CCCC(C)=CC(=O) TASK: measured log solubility in mols per litre LABEL:
       Sample llm_target: 0.3904129382012304

📥 Loading Lipophilicity ...
   ✅  lipophilicity saved → ./olmo_llm_datasets/regression/lipophilicity.csv
       train=3,360 | valid=420 | test=420 rows
       Sample llm_input : SMILES: CC(C

In [4]:
# ── CELL 1: Installs ─────────────────────────────────────────
!pip install "transformers==4.44.0" huggingface_hub
!pip install ai2-olmo>=0.3.0
!pip install hf_olmo
!pip install peft deepchem rdkit-pypi scikit-learn accelerate

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.7/43.7 kB 3.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.5/9.5 MB 59.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 23.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 39.5 MB/s eta 0:00:00
  Attempting uninstall: huggingface_hub
    Found existing installation: huggingface_hub 1.7.1
    Uninstalling huggingface_hub-1.7.1:
      Successfully uninstalled huggingface_hub-1.7.1
  Attempting uninstall: tokenizers
    Found existing installation: tokenizers 0.22.2
    Uninstalling tokenizers-0.22.2:
      Successfully uninstalled tokenizers-0.22.2
  Attempting uninstall: transformers
    Found existing installation: transformers 5.0.0
    Uninstalling transformers-5.0.0:
      Successfully uninstalled transformers-5.0.0


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
shap 0.51.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cupy-cuda12x 14.0.1 requires numpy<2.6,>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
xarray-einstats 0.10.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which

In [1]:
import os, gc, math, time, json, warnings
import numpy as np
import pandas as pd
from pathlib import Path
from dataclasses import dataclass, asdict
from typing import List, Dict, Any, Optional, Union

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from torch.cuda.amp import autocast

from hf_olmo import OLMoForCausalLM
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
import deepchem as dc
from sklearn.metrics import (
    roc_auc_score, average_precision_score,
    mean_squared_error, mean_absolute_error, r2_score,
)

warnings.filterwarnings("ignore")
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.
Instructions for updating:
experimental_relax_shapes is deprecated, use reduce_retracing instead
wandb: WARNING W&B installed but not logged in.  Run `wandb login` or set the WANDB_API_KEY env variable.


In [2]:
DATASET_REGISTRY = {
    "zinc15"       : {
        "path"     : "./olmo_llm_datasets/pretraining/zinc15_llm.csv",
        "task_type": "pretraining",
        "description": "250K drug-like SMILES for continued pretraining",
    },
    "moses"        : {
        "path"     : "./olmo_llm_datasets/generation/moses_llm.csv",
        "task_type": "generation",
        "description": "1.9M SMILES for molecular generation",
    },
    "sider"        : {
        "path"     : "./olmo_llm_datasets/classification/sider.csv",
        "task_type": "classification",
        "description": "1,427 drugs × 27 side-effect tasks",
        "task_col" : "task",
        "label_col": "label",
    },
    "muv"          : {
        "path"     : "./olmo_llm_datasets/classification/muv.csv",
        "task_type": "classification",
        "description": "93,087 molecules × 17 virtual screening tasks",
        "task_col" : "task",
        "label_col": "label",
    },
    "esol"         : {
        "path"     : "./olmo_llm_datasets/regression/esol.csv",
        "task_type": "regression",
        "description": "1,128 molecules — water solubility (log mol/L)",
        "task_col" : "task",
        "label_col": "label",
    },
    "lipophilicity": {
        "path"     : "./olmo_llm_datasets/regression/lipophilicity.csv",
        "task_type": "regression",
        "description": "4,200 molecules — lipophilicity (logD)",
        "task_col" : "task",
        "label_col": "label",
    },
}


In [3]:
def _load_csv(name: str, split: Optional[str] = None,
              task: Optional[str] = None,
              max_rows: Optional[int] = None) -> pd.DataFrame:
    """
    Load a dataset CSV by registry name.
    Optionally filter by split (train/valid/test) and task name.
    """
    meta = DATASET_REGISTRY[name]
    path = meta["path"]
    assert os.path.exists(path), (
        f"Dataset file not found: {path}\n"
        f"Run the dataset preparation script first."
    )
    df = pd.read_csv(path)

    if split and "split" in df.columns:
        df = df[df["split"] == split].reset_index(drop=True)

    if task and "task" in df.columns:
        df = df[df["task"] == task].reset_index(drop=True)

    if max_rows and len(df) > max_rows:
        df = df.sample(max_rows, random_state=42).reset_index(drop=True)

    return df


def _get_splits(name: str, task: Optional[str] = None,
                max_train: Optional[int] = None):
    """Return (train_df, valid_df, test_df) for a dataset."""
    meta = DATASET_REGISTRY[name]

    # datasets with explicit split column
    if meta["task_type"] in ("generation", "pretraining"):
        df = _load_csv(name)
        if "split" in df.columns:
            tr = df[df["split"] == "train"].reset_index(drop=True)
            te = df[df["split"] == "test"].reset_index(drop=True)
            va = te.iloc[:len(te)//2].reset_index(drop=True)
        else:
            n  = len(df)
            tr = df.iloc[:int(0.9*n)].reset_index(drop=True)
            va = df.iloc[int(0.9*n):int(0.95*n)].reset_index(drop=True)
            te = df.iloc[int(0.95*n):].reset_index(drop=True)
    else:
        tr = _load_csv(name, split="train", task=task)
        va = _load_csv(name, split="valid", task=task)
        te = _load_csv(name, split="test",  task=task)

    if max_train and len(tr) > max_train:
        tr = tr.sample(max_train, random_state=42).reset_index(drop=True)

    return tr, va, te


class _MolDataset(Dataset):
    """Tokenizes llm_input column for any task type."""
    def __init__(self, df, tokenizer, max_len, task_type):
        self.df        = df.reset_index(drop=True)
        self.tokenizer = tokenizer
        self.max_len   = max_len
        self.task_type = task_type

    def __len__(self): return len(self.df)

    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        enc = self.tokenizer(
            str(row["llm_input"]),
            max_length            = self.max_len,
            padding               = "max_length",
            truncation            = True,
            return_tensors        = "pt",
            return_token_type_ids = False,
        )
        item = {
            "input_ids"     : enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
        }
        if self.task_type in ("generation", "pretraining"):
            item["labels"] = item["input_ids"].clone()
        elif "label" in row:
            item["labels"] = torch.tensor(float(row["label"]), dtype=torch.float)
        return item

In [4]:
class OLMoAPI:
    """
    Clean ML API for OLMo-1B molecular property prediction.

    Supported tasks
    ---------------
    pretraining    : zinc15
    generation     : moses
    classification : sider, muv
    regression     : esol, lipophilicity

    Quick start
    -----------
    api = OLMoAPI()
    api.list_datasets()

    # Train on a dataset
    api.train("esol", epochs=3)
    api.train("sider", task="Hepatobiliary disorders", epochs=2)

    # Predict on new SMILES
    preds = api.predict(["CC(=O)O", "c1ccccc1"], dataset="esol")

    # Evaluate on test split
    metrics = api.evaluate("esol")

    # Generate new molecules
    smiles = api.generate(n=20, temperature=0.9)

    # Check current state
    api.status()
    """

    def __init__(
        self,
        model_name  : str = "allenai/OLMo-1B",
        lora_rank   : int = 8,
        batch_size  : int = 4,
        max_length  : int = 128,
        learning_rate: float = 2e-5,
        save_dir    : str = "./olmo_api_checkpoints",
    ):
        self.model_name    = model_name
        self.lora_rank     = lora_rank
        self.batch_size    = batch_size
        self.max_length    = max_length
        self.learning_rate = learning_rate
        self.save_dir      = save_dir
        self._trained_on   : Dict[str, Dict] = {}   # training history per dataset
        self._current_task_type = None
        self._task_head    = None

        os.makedirs(save_dir, exist_ok=True)

        print(f"Initialising OLMoAPI")
        print(f"  model      : {model_name}")
        print(f"  device     : {DEVICE}")
        print(f"  lora_rank  : {lora_rank}")
        print(f"  batch_size : {batch_size}")

        # Load tokenizer once
        self._tokenizer = AutoTokenizer.from_pretrained(model_name)
        if self._tokenizer.pad_token is None:
            self._tokenizer.pad_token = self._tokenizer.eos_token

        # Load backbone once -- shared across all tasks
        self._backbone = OLMoForCausalLM.from_pretrained(
            model_name,
            torch_dtype=torch.bfloat16,
        ).to(DEVICE)
        self._backbone.gradient_checkpointing_enable()
        self._apply_lora(lora_rank)

        total     = sum(p.numel() for p in self._backbone.parameters()) / 1e6
        trainable = sum(p.numel() for p in self._backbone.parameters() if p.requires_grad) / 1e6
        print(f"  params     : {total:.0f}M total | {trainable:.1f}M trainable (LoRA)")
        print(f"\nAPI ready. Call api.list_datasets() to see available data.\n")

    # ── Public API ─────────────────────────────────────────────

    def list_datasets(self):
        """Print all available datasets with task type and description."""
        print(f"\n{'Dataset':<16} {'Task Type':<16} {'File exists':<14} Description")
        print("-" * 80)
        for name, meta in DATASET_REGISTRY.items():
            exists = os.path.exists(meta["path"])
            status = "YES" if exists else "MISSING"
            print(f"  {name:<14} {meta['task_type']:<16} {status:<14} {meta['description']}")
        print()

    def train(
        self,
        dataset    : str,
        task       : Optional[str] = None,
        epochs     : int  = 1,
        max_rows   : int  = 1000,
        log_steps  : int  = 50,
    ) -> Dict:
        """
        Fine-tune OLMo on a dataset.

        Parameters
        ----------
        dataset  : one of zinc15 / moses / sider / muv / esol / lipophilicity
        task     : for multi-task datasets (sider/muv), which task to train on.
                   If None, uses the first available task.
        epochs   : number of training epochs
        max_rows : subsample training set (None = full dataset)
        log_steps: print loss every N steps

        Returns
        -------
        history : list of {epoch, train_loss, val_loss}

        Examples
        --------
        api.train("esol")
        api.train("sider", task="Hepatobiliary disorders", epochs=3)
        api.train("zinc15", epochs=1, max_rows=5000)
        """
        assert dataset in DATASET_REGISTRY, \
            f"Unknown dataset: {dataset!r}. Call list_datasets() to see options."

        meta      = DATASET_REGISTRY[dataset]
        task_type = meta["task_type"]

        # For multi-task clf/reg: pick first task if not specified
        if task is None and task_type in ("classification", "regression"):
            all_tasks = pd.read_csv(meta["path"])["task"].unique().tolist()
            task = all_tasks[0]
            print(f"  task not specified -- using: {task!r}")
            print(f"  available tasks: {all_tasks[:5]}{'...' if len(all_tasks)>5 else ''}")

        print(f"\n{'='*55}")
        print(f"  TRAINING | dataset={dataset} | task_type={task_type}")
        if task:
            print(f"            task={task!r}")
        print(f"{'='*55}")

        tr, va, _ = _get_splits(dataset, task=task, max_train=max_rows)
        print(f"  train={len(tr):,} | valid={len(va):,} rows")

        # Rebuild task head if task type changed
        if task_type != self._current_task_type:
            self._build_task_head(task_type)

        history = self._train_loop(tr, va, task_type, epochs, log_steps)

        # Save checkpoint
        ckpt_path = os.path.join(self.save_dir, dataset)
        self._save(ckpt_path, task_type)

        self._trained_on[dataset] = {
            "task_type" : task_type,
            "task"      : task,
            "epochs"    : epochs,
            "history"   : history,
            "checkpoint": ckpt_path,
        }
        return history

    def predict(
        self,
        smiles    : Union[str, List[str]],
        dataset   : str = None,
        task      : str = None,
    ) -> np.ndarray:
        """
        Predict molecular properties for one or more SMILES strings.

        Parameters
        ----------
        smiles  : single SMILES string or list of SMILES
        dataset : dataset name to determine task type (e.g. 'esol', 'sider')
                  If None, uses the last trained dataset.
        task    : task name for the llm_input prompt (e.g. 'measured log solubility...')

        Returns
        -------
        np.ndarray of predictions (floats for regression, 0-1 for classification)

        Examples
        --------
        api.predict("CC(=O)O", dataset="esol")
        api.predict(["CC(=O)O", "c1ccccc1"], dataset="sider",
                    task="Hepatobiliary disorders")
        """
        if isinstance(smiles, str):
            smiles = [smiles]

        # Determine task type
        task_type = self._current_task_type
        if dataset:
            task_type = DATASET_REGISTRY[dataset]["task_type"]

        assert task_type in ("classification", "regression"), \
            "predict() is for classification/regression. Use generate() for molecule generation."

        # Build prompt for each SMILES
        if task is None and dataset:
            meta = DATASET_REGISTRY[dataset]
            df_sample = pd.read_csv(meta["path"], nrows=1)
            task = df_sample["task"].iloc[0] if "task" in df_sample.columns else "property"

        rows = [{"llm_input": f"SMILES: {s} TASK: {task} LABEL:"} for s in smiles]
        df   = pd.DataFrame(rows)

        return self._predict_df(df, task_type)

    def evaluate(
        self,
        dataset : str,
        task    : str = None,
    ) -> Dict[str, float]:
        """
        Evaluate the model on the test split of a dataset.

        Parameters
        ----------
        dataset : dataset name
        task    : for multi-task datasets, which task to evaluate

        Returns
        -------
        dict of metrics:
          classification -> roc_auc, avg_precision, accuracy
          regression     -> rmse, mae, r2

        Examples
        --------
        api.evaluate("esol")
        api.evaluate("sider", task="Hepatobiliary disorders")
        """
        assert dataset in DATASET_REGISTRY, f"Unknown dataset: {dataset!r}"
        meta      = DATASET_REGISTRY[dataset]
        task_type = meta["task_type"]

        assert task_type in ("classification", "regression"), \
            "evaluate() only works for classification/regression datasets."

        if task is None:
            task = pd.read_csv(meta["path"])["task"].unique()[0]

        _, _, te = _get_splits(dataset, task=task)
        print(f"\nEvaluating {dataset!r} (task={task!r}) on {len(te):,} test samples ...")

        preds  = self._predict_df(te, task_type)
        labels = te["label"].values.astype(float)
        metrics= {}

        if task_type == "classification":
            metrics["roc_auc"]       = roc_auc_score(labels, preds)
            metrics["avg_precision"] = average_precision_score(labels, preds)
            metrics["accuracy"]      = ((preds > 0.5).astype(int) == labels.astype(int)).mean()
        elif task_type == "regression":
            metrics["rmse"] = math.sqrt(mean_squared_error(labels, preds))
            metrics["mae"]  = mean_absolute_error(labels, preds)
            metrics["r2"]   = r2_score(labels, preds)

        print(f"\n  Evaluation Results [{dataset}]:")
        for k, v in metrics.items():
            print(f"    {k:<20}: {v:.4f}")
        return metrics

    def generate(
        self,
        n           : int   = 10,
        prompt      : str   = "Generate molecule:",
        temperature : float = 0.9,
        top_p       : float = 0.95,
        max_tokens  : int   = 100,
    ) -> List[str]:
        """
        Generate new SMILES strings autoregressively.

        Parameters
        ----------
        n           : number of molecules to generate
        prompt      : generation prompt
        temperature : sampling temperature (higher = more random)
        top_p       : nucleus sampling cutoff
        max_tokens  : max new tokens per molecule

        Returns
        -------
        list of generated SMILES strings

        Examples
        --------
        api.generate(n=20)
        api.generate(n=5, prompt="Generate molecule: CC(", temperature=0.7)
        """
        self._backbone.eval()
        outputs = []
        print(f"Generating {n} molecules ...")

        for _ in range(n):
            enc = self._tokenizer(
                prompt,
                return_tensors        = "pt",
                truncation            = True,
                max_length            = self.max_length,
                return_token_type_ids = False,
            ).to(DEVICE)
            with torch.no_grad():
                gen = self._backbone.generate(
                    **enc,
                    max_new_tokens = max_tokens,
                    do_sample      = True,
                    temperature    = temperature,
                    top_p          = top_p,
                    top_k          = 50,
                    pad_token_id   = self._tokenizer.eos_token_id,
                )
            decoded = self._tokenizer.decode(gen[0], skip_special_tokens=True)
            outputs.append(decoded)

        print(f"Generated {len(outputs)} molecules")
        for i, s in enumerate(outputs[:5]):
            print(f"  [{i+1}] {s[:80]}")
        if len(outputs) > 5:
            print(f"  ... and {len(outputs)-5} more")
        return outputs

    def status(self):
        """Print current API state -- what has been trained, metrics, checkpoints."""
        print(f"\n{'='*55}")
        print(f"  OLMoAPI Status")
        print(f"{'='*55}")
        print(f"  Model      : {self.model_name}")
        print(f"  Device     : {DEVICE}")
        print(f"  Current task type: {self._current_task_type or 'none'}")

        if not self._trained_on:
            print(f"  Trained on : nothing yet -- call api.train('esol')")
        else:
            print(f"\n  Trained datasets:")
            for name, info in self._trained_on.items():
                last = info["history"][-1]
                print(f"    {name:<16} task_type={info['task_type']:<16} "
                      f"epochs={info['epochs']} "
                      f"val_loss={last['val_loss']:.4f}")
        print()

    def save(self, path: str = None):
        """Save current model state to disk."""
        path = path or os.path.join(self.save_dir, "final")
        self._save(path, self._current_task_type)
        print(f"Saved to {path}")

    def load(self, path: str):
        """Load model state from a checkpoint directory."""
        self._backbone = OLMoForCausalLM.from_pretrained(
            path, torch_dtype=torch.bfloat16
        ).to(DEVICE)
        self._tokenizer = AutoTokenizer.from_pretrained(path)
        head_path = os.path.join(path, "task_head.pt")
        if os.path.exists(head_path) and self._task_head:
            self._task_head.load_state_dict(
                torch.load(head_path, map_location=DEVICE)
            )
        print(f"Loaded from {path}")

    # ── Internal methods ───────────────────────────────────────

    def _build_task_head(self, task_type: str):
        """Build or replace the task-specific head."""
        hidden = self._backbone.config.hidden_size
        if task_type in ("classification", "regression"):
            self._task_head = nn.Sequential(
                nn.LayerNorm(hidden),
                nn.Dropout(0.1),
                nn.Linear(hidden, 1),
            ).to(DEVICE)
            print(f"  Built task head: Linear({hidden} -> 1) for {task_type}")
        else:
            self._task_head = None
        self._current_task_type = task_type

    def _train_loop(self, tr, va, task_type, epochs, log_steps):
        criterion = {
            "classification": nn.BCEWithLogitsLoss(),
            "regression"    : nn.MSELoss(),
            "generation"    : None,
            "pretraining"   : None,
        }[task_type]

        params = list(self._backbone.parameters())
        if self._task_head:
            params += list(self._task_head.parameters())
        optimizer    = AdamW(params, lr=self.learning_rate, weight_decay=0.01)

        tr_ds = _MolDataset(tr, self._tokenizer, self.max_length, task_type)
        va_ds = _MolDataset(va, self._tokenizer, self.max_length, task_type)
        tr_dl = DataLoader(tr_ds, batch_size=self.batch_size,
                           shuffle=True, num_workers=2, pin_memory=True)
        va_dl = DataLoader(va_ds, batch_size=self.batch_size,
                           shuffle=False, num_workers=2, pin_memory=True)

        total_steps  = len(tr_dl) * epochs
        warmup_steps = max(1, int(total_steps * 0.06))
        scheduler    = get_linear_schedule_with_warmup(
            optimizer, warmup_steps, total_steps
        )

        print(f"  steps={total_steps:,} | warmup={warmup_steps} | "
              f"lr={self.learning_rate} | batch={self.batch_size}\n")

        history = []
        for epoch in range(epochs):
            # Train
            self._backbone.train()
            if self._task_head: self._task_head.train()
            tr_loss, n = 0.0, 0

            for step, batch in enumerate(tr_dl):
                batch = {k: v.to(DEVICE) for k, v in batch.items()}
                with autocast(dtype=torch.bfloat16, enabled=(DEVICE=="cuda")):
                    loss = self._forward(batch, task_type, criterion)

                optimizer.zero_grad()
                loss.backward()
                nn.utils.clip_grad_norm_(params, 1.0)
                optimizer.step()
                scheduler.step()
                tr_loss += loss.item(); n += 1

                if step % log_steps == 0:
                    lr = scheduler.get_last_lr()[0]
                    print(f"    step {step:>4}/{len(tr_dl)} | "
                          f"loss={loss.item():.4f} | lr={lr:.2e}")

            # Validate
            self._backbone.eval()
            if self._task_head: self._task_head.eval()
            va_loss, vn = 0.0, 0
            with torch.no_grad():
                for batch in va_dl:
                    batch = {k: v.to(DEVICE) for k, v in batch.items()}
                    with autocast(dtype=torch.bfloat16, enabled=(DEVICE=="cuda")):
                        va_loss += self._forward(batch, task_type, criterion).item()
                    vn += 1

            log = {
                "epoch"      : epoch + 1,
                "train_loss" : tr_loss / max(n, 1),
                "val_loss"   : va_loss / max(vn, 1),
            }
            history.append(log)
            print(f"\n  Epoch {epoch+1}/{epochs} | "
                  f"train={log['train_loss']:.4f} | val={log['val_loss']:.4f}\n")

        return history

    def _forward(self, batch, task_type, criterion):
        if task_type in ("generation", "pretraining"):
            return self._backbone(
                input_ids      = batch["input_ids"],
                attention_mask = batch["attention_mask"],
                labels         = batch["labels"],
            ).loss
        else:
            out = self._backbone(
                input_ids            = batch["input_ids"],
                attention_mask       = batch["attention_mask"],
                output_hidden_states = True,
            )
            seq_lens = batch["attention_mask"].sum(dim=1) - 1
            mol_repr = out.hidden_states[-1][
                torch.arange(out.hidden_states[-1].size(0)), seq_lens
            ]
            logits = self._task_head(mol_repr).squeeze(-1)
            return criterion(logits, batch["labels"].to(logits.dtype))

    @torch.no_grad()
    def _predict_df(self, df, task_type) -> np.ndarray:
        self._backbone.eval()
        if self._task_head: self._task_head.eval()
        ds     = _MolDataset(df, self._tokenizer, self.max_length, task_type)
        loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False)
        preds  = []
        for batch in loader:
            batch = {k: v.to(DEVICE) for k, v in batch.items()}
            with autocast(dtype=torch.bfloat16, enabled=(DEVICE=="cuda")):
                out = self._backbone(
                    input_ids            = batch["input_ids"],
                    attention_mask       = batch["attention_mask"],
                    output_hidden_states = True,
                )
            seq_lens = batch["attention_mask"].sum(dim=1) - 1
            mol_repr = out.hidden_states[-1][
                torch.arange(out.hidden_states[-1].size(0)), seq_lens
            ]
            logits = self._task_head(mol_repr).squeeze(-1)
            p = torch.sigmoid(logits) if task_type == "classification" else logits
            preds.append(p.cpu().float().numpy())
        return np.concatenate(preds, axis=0)

    def _save(self, path, task_type):
        os.makedirs(path, exist_ok=True)
        self._backbone.save_pretrained(path)
        self._tokenizer.save_pretrained(path)
        if self._task_head:
            torch.save(self._task_head.state_dict(),
                       os.path.join(path, "task_head.pt"))
        meta = {"task_type": task_type, "model_name": self.model_name}
        json.dump(meta, open(os.path.join(path, "api_meta.json"), "w"))
        print(f"  Checkpoint -> {path}")

    def _apply_lora(self, rank):
        if rank == 0:
            return
        try:
            from peft import get_peft_model, LoraConfig, TaskType
            linear_names = {
                name.split(".")[-1]
                for name, mod in self._backbone.named_modules()
                if isinstance(mod, nn.Linear)
            }
            linear_names.discard("lm_head")
            lora_cfg = LoraConfig(
                task_type      = TaskType.CAUSAL_LM,
                r              = rank,
                lora_alpha     = rank * 2,
                lora_dropout   = 0.05,
                bias           = "none",
                target_modules = list(linear_names),
            )
            self._backbone = get_peft_model(self._backbone, lora_cfg)
            trainable = sum(p.numel() for p in self._backbone.parameters()
                            if p.requires_grad)
            total = sum(p.numel() for p in self._backbone.parameters())
            print(f"  LoRA rank={rank} | "
                  f"trainable: {trainable/1e6:.1f}M / {total/1e6:.0f}M")
        except ImportError:
            print("  peft not installed -- pip install peft")

In [5]:
api = OLMoAPI(
    lora_rank    = 8,
    batch_size   = 4,
    learning_rate= 2e-5,
)

Initialising OLMoAPI
  model      : allenai/OLMo-1B
  device     : cuda
  lora_rank  : 8
  batch_size : 4


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/65.0 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/4.71G [00:00<?, ?B/s]

  LoRA rank=8 | trainable: 5.2M / 1182M
  params     : 1182M total | 5.2M trainable (LoRA)

API ready. Call api.list_datasets() to see available data.



In [9]:
# ── PATCH CELL: fixes validation hang + task head bug ────────
import types, torch.nn as nn

# ── Fix 1: _get_splits with capped valid ─────────────────────
def _get_splits_fixed(name, task=None, max_train=None, max_valid=500):
    meta = DATASET_REGISTRY[name]

    if meta["task_type"] in ("generation", "pretraining"):
        df = pd.read_csv(meta["path"])
        if "split" in df.columns:
            tr = df[df["split"] == "train"].reset_index(drop=True)
            te = df[df["split"] == "test"].reset_index(drop=True)
            va = te.iloc[:len(te)//2].reset_index(drop=True)
        else:
            n  = len(df)
            tr = df.iloc[:int(0.9*n)].reset_index(drop=True)
            va = df.iloc[int(0.9*n):int(0.95*n)].reset_index(drop=True)
            te = df.iloc[int(0.95*n):].reset_index(drop=True)
    else:
        tr = _load_csv(name, split="train", task=task)
        va = _load_csv(name, split="valid", task=task)
        te = _load_csv(name, split="test",  task=task)

    if max_train and len(tr) > max_train:
        tr = tr.sample(max_train, random_state=42).reset_index(drop=True)
    if max_valid and len(va) > max_valid:
        va = va.sample(max_valid, random_state=42).reset_index(drop=True)

    return tr, va, te


# ── Fix 2: train() uses capped valid ─────────────────────────
def train_fixed(self, dataset, task=None, epochs=1, max_rows=1000, log_steps=50):
    assert dataset in DATASET_REGISTRY, \
        f"Unknown dataset: {dataset!r}. Call list_datasets() to see options."
    meta      = DATASET_REGISTRY[dataset]
    task_type = meta["task_type"]

    if task is None and task_type in ("classification", "regression"):
        all_tasks = pd.read_csv(meta["path"])["task"].unique().tolist()
        task = all_tasks[0]
        print(f"  task not specified -- using: {task!r}")
        print(f"  available tasks: {all_tasks[:5]}{'...' if len(all_tasks)>5 else ''}")

    print(f"\n{'='*55}")
    print(f"  TRAINING | dataset={dataset} | task_type={task_type}")
    if task: print(f"            task={task!r}")
    print(f"{'='*55}")

    tr, va, _ = _get_splits_fixed(
        dataset, task=task,
        max_train=max_rows,
        max_valid=500,
    )
    print(f"  train={len(tr):,} | valid={len(va):,} rows")

    if task_type != self._current_task_type:
        self._build_task_head(task_type)

    history = self._train_loop(tr, va, task_type, epochs, log_steps)

    ckpt_path = os.path.join(self.save_dir, dataset)
    self._save(ckpt_path, task_type)

    self._trained_on[dataset] = {
        "task_type" : task_type,
        "task"      : task,
        "epochs"    : epochs,
        "history"   : history,
        "checkpoint": ckpt_path,
    }
    return history


# ── Fix 3: _predict_df rebuilds task head if None ────────────
@torch.no_grad()
def _predict_df_fixed(self, df, task_type):
    if self._task_head is None and task_type in ("classification", "regression"):
        print(f"  Rebuilding task head for {task_type} ...")
        hidden = self._backbone.config.hidden_size
        self._task_head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(0.1),
            nn.Linear(hidden, 1),
        ).to(DEVICE)
        self._current_task_type = task_type

    self._backbone.eval()
    if self._task_head: self._task_head.eval()

    ds     = _MolDataset(df, self._tokenizer, self.max_length, task_type)
    loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False)
    preds  = []

    for batch in loader:
        batch = {k: v.to(DEVICE) for k, v in batch.items()}
        with autocast(dtype=torch.bfloat16, enabled=(DEVICE == "cuda")):
            out = self._backbone(
                input_ids            = batch["input_ids"],
                attention_mask       = batch["attention_mask"],
                output_hidden_states = True,
            )
        seq_lens = batch["attention_mask"].sum(dim=1) - 1
        mol_repr = out.hidden_states[-1][
            torch.arange(out.hidden_states[-1].size(0)), seq_lens
        ]
        logits = self._task_head(mol_repr).squeeze(-1)
        p = torch.sigmoid(logits) if task_type == "classification" else logits
        preds.append(p.cpu().float().numpy())

    return np.concatenate(preds, axis=0)


# ── Apply all 3 patches to live api object ────────────────────
api.train        = types.MethodType(train_fixed,       api)
api._predict_df  = types.MethodType(_predict_df_fixed, api)
print("✓ Fix 1: _get_splits now caps valid=500 rows")
print("✓ Fix 2: train() uses capped valid")
print("✓ Fix 3: _predict_df rebuilds task head if None")
print()
print("Now run:")
print("  api.train('moses', epochs=1, max_rows=500)")

✓ Fix 1: _get_splits now caps valid=500 rows
✓ Fix 2: train() uses capped valid
✓ Fix 3: _predict_df rebuilds task head if None

Now run:
  api.train('moses', epochs=1, max_rows=500)


In [7]:
api.list_datasets()


Dataset          Task Type        File exists    Description
--------------------------------------------------------------------------------
  zinc15         pretraining      YES            250K drug-like SMILES for continued pretraining
  moses          generation       YES            1.9M SMILES for molecular generation
  sider          classification   YES            1,427 drugs × 27 side-effect tasks
  muv            classification   YES            93,087 molecules × 17 virtual screening tasks
  esol           regression       YES            1,128 molecules — water solubility (log mol/L)
  lipophilicity  regression       YES            4,200 molecules — lipophilicity (logD)



In [12]:
api.train("moses",epochs=1, max_rows=1000)


  TRAINING | dataset=moses | task_type=generation
  train=1,000 | valid=500 rows
  steps=250 | warmup=15 | lr=2e-05 | batch=4

    step    0/250 | loss=0.2868 | lr=1.33e-06
    step   50/250 | loss=0.2308 | lr=1.69e-05
    step  100/250 | loss=0.1972 | lr=1.27e-05
    step  150/250 | loss=0.2860 | lr=8.43e-06
    step  200/250 | loss=0.1759 | lr=4.17e-06

  Epoch 1/1 | train=0.2325 | val=0.2253

  Checkpoint -> ./olmo_api_checkpoints/moses


[{'epoch': 1,
  'train_loss': 0.23248139202594756,
  'val_loss': 0.2252972048521042}]

In [14]:
new_mols = api.generate(
    n           = 10,
    prompt      = "Generate molecule:",
    temperature = 0.9,
    top_p       = 0.95,
    max_tokens  = 100,
)

new_mols_2 = api.generate(
    n           = 5,
    prompt      = "Generate molecule: CC(",   # partial SMILES seed
    temperature = 0.7,                        # lower = more focused
)

new_mols_3 = api.generate(
    n           = 5,
    prompt      = "Generate molecule: c1ccc(", # aromatic seed
    temperature = 1.0,                         # higher = more diverse
)

Generating 10 molecules ...
Generated 10 molecules
  [1] Generate molecule: CCn1c(C(=O)NCc2ccccc2)n(Cc3noc(C)n3)c1enalichelenalicenatalen
  [2] Generate molecule: Cc1cc(N2CCC(c3cccn3)CC2)cc(C)c1(Cl)c1ccccc1(Cl)c1ccccc1Cl(Cl)
  [3] Generate molecule: Cc1cccc(-c2ccc(S(=O)(=O)N3CCCO3)cc2)c1Cretnterretretretretret
  [4] Generate molecule: COc1ccc(CCNC(=O)c2ccc(F)cc2)cc1tuttilisatticattisoc2ccccc21tu
  [5] Generate molecule: COc1cccc(NC(=O)c2ccccc2)c1C:s1:

where:
c1cccc(C(=O)N2C(=O)NC
  ... and 5 more
Generating 5 molecules ...
Generated 5 molecules
  [1] Generate molecule: CC(C)S(=O)(=O)N1CCCCC1c1ccccc1rettretrettrettretrettrettrettr
  [2] Generate molecule: CC(C)C(=O)c1ccccc1-c1ccc(C#N)cc1teilenet-2teilenet-3frettentr
  [3] Generate molecule: CC(C)c1cccc(NC(=O)c2ccc(F)cc2)c1trettlettencacacacacacacacaca
  [4] Generate molecule: CC(=O)NCCc1ncc2n1CCC(=O)C2retrievable-21-july-2010-17-16-5-5-
  [5] Generate molecule: CC(=O)c1ccc(SCC(=O)Nc2cc3cccnc3c2)cc1enici-n1cc2cccnc2c2ccccc
Generating 5 

In [15]:
print("\nAll generated molecules:")
for i, s in enumerate(new_mols + new_mols_2 + new_mols_3, 1):
    print(f"  [{i:>2}] {s[:80]}")


All generated molecules:
  [ 1] Generate molecule: CCn1c(C(=O)NCc2ccccc2)n(Cc3noc(C)n3)c1enalichelenalicenatalen
  [ 2] Generate molecule: Cc1cc(N2CCC(c3cccn3)CC2)cc(C)c1(Cl)c1ccccc1(Cl)c1ccccc1Cl(Cl)
  [ 3] Generate molecule: Cc1cccc(-c2ccc(S(=O)(=O)N3CCCO3)cc2)c1Cretnterretretretretret
  [ 4] Generate molecule: COc1ccc(CCNC(=O)c2ccc(F)cc2)cc1tuttilisatticattisoc2ccccc21tu
  [ 5] Generate molecule: COc1cccc(NC(=O)c2ccccc2)c1C:s1:

where:
c1cccc(C(=O)N2C(=O)NC
  [ 6] Generate molecule: Nc1cc(OC(=O)c2ccc(C(=O)N3CCC3)c(F)c2)nn1. molecule:C(=O)c1ccc
  [ 7] Generate molecule: COc1ccc(CNC(=O)NCc2nc3ccccc3C2)cc1chigidaeididiciidaeididicii
  [ 8] Generate molecule: O=C(c1ccncc1)N1CCC2(c1ccccc1)N(C)C1C. molecule: O=C(c1ccncc1)
  [ 9] Generate molecule: CCOC(=O)NC(=O)c1cc(-c2ccc(F)cc2)ccc1Nc2ccc(F)c(F)c2OCc1cccc2c
  [10] Generate molecule: CCCn1nc(C(=O)NCc2ccco2)c(C)C1teal. molecule: Cc1noc(C)c1ccc2n
  [11] Generate molecule: CC(C)S(=O)(=O)N1CCCCC1c1ccccc1rettretrettrettretrettrettrettr
  [12]

In [16]:
api.status()


  OLMoAPI Status
  Model      : allenai/OLMo-1B
  Device     : cuda
  Current task type: generation

  Trained datasets:
    moses            task_type=generation       epochs=1 val_loss=0.2253



In [17]:
# ── Save / Load ───────────────────────────────────────────────
api.save("./my_olmo_model")

  Checkpoint -> ./my_olmo_model
Saved to ./my_olmo_model
